# T-KGAT Stage 2 — thời gian trên cạnh KG

Dùng Dataset code/data đã đóng gói và ZIP Recovery **KGAT có KG đã hoàn tất**.
Stage 2 tự đọc cấu hình KGAT đã chọn trên validation và quét đủ
**γ = {1, 3, 10} × 5 seed 2019–2023 = 15 lượt**.
Stage 1 và Stage 2 có thể chạy song song sau khi KGAT đã hoàn thành.

Thời gian được áp dụng lên cạnh KG có ngày. Ngày KG của bộ Last-FM này được suy ra
từ Freebase. Fact không có ngày nhận phạt thời gian bằng 0; độ cũ chuẩn hóa được
chặn tối đa ở 1. Trong payload Last-FM hiện tại, toàn bộ 125.640 fact có ngày
đều bị chặn ở 1. Vì vậy Stage 2 trên bộ này chủ yếu kiểm tra tác động giảm
trọng số nhóm cạnh có ngày so với nhóm không có ngày; các fact có ngày cùng
nhận mức phạt gamma trước softmax. Cạnh user–item nhận mức phạt bằng 0.

Mặc định chạy 2 lượt mới mỗi phiên và lưu ZIP sau mỗi lượt hoàn tất.
Chọn **Save Version → Save & Run All**, bật GPU T4 và kiểm tra log
`Running tkgat Stage 2 only` trước khi đóng tab/tắt máy.

In [ ]:
from pathlib import Path

PAYLOAD_DIR = None
# Select a ZIP or results directory if automatic discovery finds several recoveries.
RESTORE_SOURCE = None
PROJECT_ROOT = Path('/kaggle/working/t-kgat-stage2')
RUN_TRAINING = True
NEW_TKGAT_FITS_THIS_SESSION = 2
CHECKPOINT_ZIP = Path('/kaggle/working/tkgat-lastfm-stage2-results.zip')

## 1. Nạp code/data và khôi phục KGAT

Phiên đầu dùng ZIP `tkgat-lastfm-results.zip` có kết quả KGAT đầy đủ 20 lượt và 5 seed test.
Phiên tiếp theo dùng ZIP `tkgat-lastfm-stage2-results.zip` mới nhất của nhánh Stage 2.
Dataset code/data ban đầu đã có `kg_t1_dates.tsv` và `kg_t2_dates.tsv`.
Gắn một bản Recovery, hoặc đặt `RESTORE_SOURCE` nếu có nhiều bản trong Input.

In [ ]:
import hashlib
import json
import os
import runpy
import subprocess
import sys
from pathlib import Path, PurePosixPath
from zipfile import ZIP_DEFLATED, ZipFile, is_zipfile


def recovery_archive(input_root, working, expected_identity, explicit=None):
    """Resolve one matching ZIP or extracted results directory, never guess between copies."""
    if explicit is not None:
        location = Path(explicit)
        if not location.exists():
            raise RuntimeError(f'Recovery path does not exist: {location}')
        locations = [location]
    else:
        locations = sorted(input_root.rglob('*.zip'))
        locations += sorted(p.parent for p in input_root.rglob('kaggle-run.json')
                            if p.parent.name == 'results')
    matches = []
    inspected = []
    for location in locations:
        if location.is_file() and is_zipfile(location):
            with ZipFile(location) as archive:
                marker = 'results/kaggle-run.json'
                if marker not in archive.namelist():
                    continue
                saved_identity = json.loads(archive.read(marker))
            candidate = ('zip', location)
        elif location.is_dir():
            folder = location if location.name == 'results' else location / 'results'
            marker = folder / 'kaggle-run.json'
            if not marker.is_file():
                continue
            saved_identity = json.loads(marker.read_text(encoding='utf-8'))
            candidate = ('directory', folder)
        else:
            continue
        inspected.append(str(location))
        if saved_identity == expected_identity:
            matches.append(candidate)
    if not matches:
        raise RuntimeError(
            'No matching Recovery found. Add the previous results as Input; '
            'both ZIP and extracted results/ are supported. '
            f'Found result locations: {inspected}. Refusing to start Stage 2 without KGAT Recovery.'
        )
    if len(matches) != 1:
        raise RuntimeError(f'Multiple recoveries match: {matches}. Set RESTORE_SOURCE explicitly.')
    kind, source = matches[0]
    print('Recovery selected:', source, flush=True)
    if kind == 'zip':
        return source
    target = working / 'kgat-recovery-normalized.zip'
    with ZipFile(target, 'w', compression=ZIP_DEFLATED) as archive:
        for folder in (source, source.parent / 'reports'):
            if folder.is_dir():
                for path in sorted(folder.rglob('*')):
                    if path.is_file():
                        archive.write(path, path.relative_to(source.parent).as_posix())
    return target


def require_stage2_inputs(suite):
    """Validate the completed KGAT dependency and the full Stage 2 gamma/seed grid."""
    if not suite['complete']('kgat'):
        raise RuntimeError('KGAT is incomplete in Recovery. Use the completed KGAT ZIP before Stage 2.')
    print('kgat: complete (verified)', flush=True)
    import yaml

    model_path = suite['ROOT'] / 'configs/model/tkgat_kg.yaml'
    config = yaml.safe_load(model_path.read_text(encoding='utf-8'))
    if (config.get('name') != 'tkgat' or config.get('time_on') != 'kg'
            or config.get('control') or config.get('from_kgat') != 'kgat'):
        raise RuntimeError('Expected the unpermuted Stage 2 configuration from KGAT.')
    if config.get('seeds') != list(range(2019, 2024)):
        raise RuntimeError('Stage 2 requires all five seeds 2019-2023.')
    if config.get('grid') != [{'gamma': 1}, {'gamma': 3}, {'gamma': 10}]:
        raise RuntimeError('Stage 2 requires the complete gamma grid [1, 3, 10].')
    runner = runpy.run_path(str(suite['ROOT'] / 'scripts/run.py'))
    folder = suite['ROOT'] / 'results/runs' / suite['RUN_NAME']
    points, base = runner['tkgat_points'](config, model_path, folder)
    print('Inherited KGAT configuration:', json.dumps(base, sort_keys=True), flush=True)
    print('Stage 2 grid: gamma=[1, 3, 10], seeds=[2019, 2020, 2021, 2022, 2023]; 15 fits.', flush=True)
    return points


input_root = Path('/kaggle/input')
working = Path('/kaggle/working')
if PAYLOAD_DIR is None:
    manifests = sorted(input_root.rglob('payload-manifest.json'))
    if not manifests:
        packages = sorted(input_root.rglob('tkgat-lastfm-latest-kaggle.zip'))
        if len(packages) != 1:
            raise RuntimeError('Add the original code/data Dataset as Input, or set PAYLOAD_DIR.')
        unpacked = working / 'tkgat-payload'
        with ZipFile(packages[0]) as archive:
            for name in archive.namelist():
                path = PurePosixPath(name)
                if path.is_absolute() or '..' in path.parts or '\\' in name:
                    raise ValueError(f'Invalid payload member: {name}')
            archive.extractall(unpacked)
        manifests = [unpacked / 'payload-manifest.json']
    if len({path.read_text(encoding='utf-8') for path in manifests}) != 1:
        raise RuntimeError('Multiple code/data payloads found. Set PAYLOAD_DIR.')
    payload_root = manifests[0].parent
else:
    payload_root = Path(PAYLOAD_DIR)
manifest = json.loads((payload_root / 'payload-manifest.json').read_text(encoding='utf-8'))
bundle = payload_root / 'source.bundle'
if hashlib.sha256(bundle.read_bytes()).hexdigest() != manifest['files_sha256']['source.bundle']:
    raise ValueError('Source bundle hash mismatch')
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', str(bundle), str(PROJECT_ROOT)], check=True)
commit = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
if commit != manifest['source_commit']:
    raise RuntimeError('Working copy belongs to another payload. Start a fresh session.')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.environ['PYTHONPATH'] = str(PROJECT_ROOT / 'src')
os.environ['PYTHONUNBUFFERED'] = '1'
suite = runpy.run_path(str(PROJECT_ROOT / 'scripts/kaggle_lastfm.py'))
suite['prepare'](payload_root)
has_progress = (PROJECT_ROOT / 'results/registry.csv').exists() or any(
    (PROJECT_ROOT / 'results/runs').rglob('*.pt')
)
if not has_progress:
    restore = recovery_archive(input_root, working, suite['identity'](), RESTORE_SOURCE)
    suite['extract_results'](restore)
else:
    print('Keeping existing working progress; Recovery will not overwrite it.', flush=True)
suite['record_identity']()
require_stage2_inputs(suite)
suite['backup'](CHECKPOINT_ZIP)
print('Source snapshot:', commit, flush=True)
print('Recovery verified. T-KGAT Stage 2 is the only model enabled.', flush=True)

## 2. Chạy lưới T-KGAT Stage 2

Log phải có `kgat: complete (verified)`, lưới gamma đầy đủ và
`Running tkgat Stage 2 only`. Stage 2 gọi cấu hình `tkgat_kg.yaml` với `time_on: kg`.
Mỗi lượt huấn luyện dùng train, dừng sớm theo Recall@20 validation.
Sau đủ 15 lượt, script chọn gamma có Recall@20 validation trung bình tốt nhất qua 5 seed,
rồi chấm test 5 seed của gamma được chọn.

Các lượt hoàn tất được đọc từ checkpoint. Lượt bị ngắt giữa epoch phải chạy lại từ đầu.
Giữ kết quả của từng nhánh bằng ZIP có tên riêng khi chạy trên các máy khác nhau.

In [ ]:
import torch

require_stage2_inputs(suite)
print('CUDA:', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU T4 before Save & Run All.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
if NEW_TKGAT_FITS_THIS_SESSION < 1:
    raise ValueError('NEW_TKGAT_FITS_THIS_SESSION must be positive.')

if RUN_TRAINING:
    for step in range(NEW_TKGAT_FITS_THIS_SESSION):
        if suite['complete']('tkgat_kg'):
            print('tkgat_kg: complete. No other model will be started.', flush=True)
            break
        print(f'Running tkgat Stage 2 only; fit budget {step + 1}/{NEW_TKGAT_FITS_THIS_SESSION}', flush=True)
        try:
            suite['execute']([
                'scripts/run.py', '--data', suite['DATA_CONFIG'],
                '--model', 'configs/model/tkgat_kg.yaml', '--device', 'cuda',
                '--resume', '--max-new-runs', '1',
            ])
        finally:
            suite['backup'](CHECKPOINT_ZIP)
else:
    print('Training disabled. Recovery verified only.', flush=True)

suite['backup'](CHECKPOINT_ZIP)
print('tkgat_kg:', 'complete' if suite['complete']('tkgat_kg') else 'pending (more fits needed)', flush=True)
print('Save this output:', CHECKPOINT_ZIP, flush=True)

## 3. Lưu kết quả và chạy phiên sau

Tải **`tkgat-lastfm-stage2-results.zip`** từ Output của Version đã hoàn tất.
Dùng ZIP Stage 2 mới nhất để thay Recovery cho phiên tiếp theo, giữ Dataset code/data.
Mức 2 lượt mới/phiên tương ứng tối đa 8 phiên cho lưới 15 lượt.

Khi hoàn tất, `tkgat_kg_summary.json` ghi gamma đã chọn và điểm test trung bình ±
độ lệch chuẩn mẫu qua 5 seed. ZIP cũng giữ kết quả theo từng user và checkpoint.
Đối chứng hoán vị ngày fact sử dụng gamma Stage 2 đã chọn và sẽ chạy bằng notebook riêng.